In [1]:
!pip install pypdf chromadb sentence-transformers groq requests -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
import requests

# We'll use the original "Attention Is All You Need" paper
# It's public, well-known, and perfect for testing Q&A
PDF_URL = "https://arxiv.org/pdf/1706.03762"
PDF_PATH = "attention_paper.pdf"

response = requests.get(PDF_URL)
with open(PDF_PATH, "wb") as f:
    f.write(response.content)

print(f"Downloaded PDF: {PDF_PATH}")
print(f"File size: {len(response.content) / 1024:.1f} KB")

Downloaded PDF: attention_paper.pdf
File size: 2163.3 KB


In [3]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""

    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:  # some pages may return None
            full_text += f"\n--- Page {page_num + 1} ---\n"
            full_text += text

    return full_text

raw_text = extract_text_from_pdf(PDF_PATH)

print(f"Total characters extracted: {len(raw_text)}")
print(f"Total pages: {len(PdfReader(PDF_PATH).pages)}")
print("\nFirst 500 characters preview:")
print(raw_text[:500])

Total characters extracted: 39861
Total pages: 15

First 500 characters preview:

--- Page 1 ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.tor


In [12]:
import re

def split_into_chunks(text, chunk_size=800, overlap=200):
    """
    Sentence-aware chunking:
    - Splits on sentence boundaries (not raw characters)
    - Groups sentences into ~800 char chunks
    - 200 char overlap between chunks to avoid losing context at boundaries
    """
    # Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) < chunk_size:
            current_chunk += " " + sentence
        else:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            # Start next chunk with overlap from previous
            overlap_buffer = current_chunk[-overlap:] if len(current_chunk) > overlap else current_chunk
            current_chunk = overlap_buffer + " " + sentence

    # Add the last remaining chunk
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks

chunks = split_into_chunks(raw_text, chunk_size=800, overlap=200)

print(f"Total chunks created: {len(chunks)}")
print(f"Average chunk size: {sum(len(c) for c in chunks) / len(chunks):.0f} chars")
print(f"\nSample chunk (chunk #5):")
print("-" * 40)
print(chunks[4])

Total chunks created: 78
Average chunk size: 708 chars

Sample chunk (chunk #5):
----------------------------------------
less model variants in our original codebase and
tensor2tensor. Llion also experimented with novel model variants, was responsible for our initial codebase, and
efficient inference and visualizations. Lukasz and Aidan spent countless long days designing various parts of and
implementing tensor2tensor, replacing our earlier codebase, greatly improving results and massively accelerating
our research. †Work performed while at Google Brain. ‡Work performed while at Google Research. 31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA.


In [13]:
from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2:
# - 384 dimensional embeddings
# - Very fast (runs well on CPU)
# - Trained specifically for semantic similarity tasks
# - 22M parameters — small but effective

print("Loading embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded.")

# Quick test to verify it works
test_embedding = embed_model.encode("This is a test sentence.")
print(f"Embedding shape: {test_embedding.shape}")  # Should be (384,)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.
Embedding shape: (384,)


In [14]:
import chromadb

# Persistent client saves the vector DB to disk
# So if Colab restarts, you don't re-embed everything
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Delete collection if it exists (for clean reruns)
try:
    chroma_client.delete_collection("pdf_rag")
except:
    pass

collection = chroma_client.create_collection(
    name="pdf_rag",
    metadata={"hnsw:space": "cosine"}  # cosine similarity for text
)

print("ChromaDB collection created.")
print("Embedding all chunks — this may take 1-2 minutes on CPU...")

# Embed all chunks in one batch (faster than one-by-one)
embeddings = embed_model.encode(chunks, show_progress_bar=True)

# Store in ChromaDB
# ChromaDB needs: documents (text), embeddings (vectors), ids (unique strings)
collection.add(
    documents=chunks,
    embeddings=embeddings.tolist(),
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"\nStored {collection.count()} chunks in ChromaDB.")

ChromaDB collection created.
Embedding all chunks — this may take 1-2 minutes on CPU...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


Stored 78 chunks in ChromaDB.


In [15]:
def retrieve_relevant_chunks(question, collection, embed_model, top_k=5):
    """
    Given a question:
    1. Embed the question into a vector
    2. Search ChromaDB for the top_k most similar chunk vectors
    3. Return those chunks as context
    """
    # Step 1: Embed the question
    question_embedding = embed_model.encode(question).tolist()

    # Step 2: Query ChromaDB
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )

    # Results structure:
    # results['documents'][0] → list of top_k chunk texts
    # results['distances'][0] → list of distances (lower = more similar for cosine)

    retrieved_chunks = results['documents'][0]
    distances = results['distances'][0]

    return retrieved_chunks, distances

# Test retrieval
test_q = "What is the attention mechanism?"
chunks_retrieved, scores = retrieve_relevant_chunks(test_q, collection, embed_model)

print(f"Question: {test_q}")
print(f"\nTop 3 retrieved chunks:")
for i, (chunk, score) in enumerate(zip(chunks_retrieved[:3], scores[:3])):
    print(f"\n[Chunk {i+1} | Distance: {score:.4f}]")
    print(chunk[:200] + "...")

Question: What is the attention mechanism?

Top 3 retrieved chunks:

[Chunk 1 | Distance: 0.4190]
pplication
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
. <EOS>
<pad>
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
....

[Chunk 2 | Distance: 0.4345]
ions. This
masking, combined with fact that the output embeddings are offset by one position, ensures that the
predictions for position i can depend only on the known outputs at positions less than i....

[Chunk 3 | Distance: 0.4749]
proach we take in our model. As side benefit, self-attention could yield more interpretable models. We inspect attention distributions
from our models and present and discuss examples in the appendix....


In [19]:
from groq import Groq
from google.colab import userdata

# Store your key in Colab Secrets (left panel → key icon → name it GROQ_API_KEY)
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

groq_client = Groq(api_key=GROQ_API_KEY)

def generate_answer(question, context_chunks, groq_client):
    # Filter out garbage chunks (too many single characters = extraction artifact)
    clean_chunks = [c for c in context_chunks if len(c.split()) > 15]

    if not clean_chunks:
        return "Could not find relevant context in the document."

    context = "\n\n---\n\n".join(clean_chunks)

    prompt = f"""You are a helpful assistant answering questions about the research paper "Attention Is All You Need".
Answer the question using the provided context.
Extract and summarize the relevant information even if it is not stated directly.
If truly no relevant information exists in the context, say "I don't have enough information."


Context:
{context}

Question: {question}

Answer:"""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=512
    )

    return response.choices[0].message.content.strip()

result = rag_query(
    "What problem does the Transformer architecture solve?",
    collection, embed_model, groq_client
)

print("QUESTION:", result["question"])
print("\nANSWER:", result["answer"])

QUESTION: What problem does the Transformer architecture solve?

ANSWER: The Transformer architecture solves the problem of sequential computation in neural networks, particularly in tasks such as machine translation, where the number of operations required to relate signals from two arbitrary input or output positions grows linearly or logarithmically with the distance between positions. This makes it more difficult to learn dependencies between distant positions.

The Transformer architecture reduces this problem by using self-attention and multi-head attention mechanisms, which allow for parallelization and a constant number of operations to relate signals from any two positions in the input or output sequence. This enables the model to learn dependencies between distant positions more effectively.

In the context of machine translation, the Transformer architecture allows for significantly more parallelization and can reach a new state of the art in translation quality after being 

In [20]:
def rag_query(question, collection, embed_model, groq_client, top_k=5):
    """
    End-to-end RAG pipeline:
    Question → Retrieve → Generate → Return Answer + Sources
    """
    # Step 1: Retrieve
    retrieved_chunks, distances = retrieve_relevant_chunks(
        question, collection, embed_model, top_k
    )

    # Step 2: Generate
    answer = generate_answer(question, retrieved_chunks, groq_client)

    return {
        "question": question,
        "answer": answer,
        "source_chunks": retrieved_chunks,
        "retrieval_scores": distances
    }

# Test the full pipeline
result = rag_query(
    "What problem does the Transformer architecture solve?",
    collection, embed_model, groq_client
)

print("QUESTION:", result["question"])
print("\nANSWER:", result["answer"])
print("\nRETRIEVAL SCORES:", [f"{s:.4f}" for s in result["retrieval_scores"]])

QUESTION: What problem does the Transformer architecture solve?

ANSWER: The Transformer architecture solves the problem of sequential computation in neural networks, particularly in tasks such as machine translation. The traditional sequential computation approach makes it difficult to learn dependencies between distant positions in a sequence. The Transformer architecture reduces this problem by using self-attention and multi-head attention mechanisms, which allow for parallelization and enable the model to learn dependencies between distant positions more efficiently.

In the context of machine translation, the Transformer architecture allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs. This is a significant improvement over previous models that used convolutional neural networks as basic building blocks, which had limitations in learning dependencies between d

In [21]:
evaluation_questions = [
    # Factual
    "What is the title of the paper?",
    "Who are the authors of this paper?",
    "What does BLEU score measure?",
    "How many attention heads are used in the base model?",
    "What is the model dimension d_model used in the base model?",
    # Conceptual
    "What is the main contribution of the Transformer model?",
    "What problem does the Transformer architecture solve?",
    "What is the purpose of positional encoding?",
    "What is multi-head attention?",
    "What is the difference between encoder and decoder in the Transformer?",
    # Specific details
    "How many encoder layers does the Transformer have?",
    "What optimizer was used to train the model?",
    "What dataset was used for English-to-German translation?",
    "What BLEU score did the Transformer achieve on English-German translation?",
    "What hardware was used to train the model?",
    # Harder
    "Why is self-attention better than recurrent layers for sequence tasks?",
    "What is the role of the feed-forward network in each layer?",
    "How does the model handle variable length sequences?",
    "What is label smoothing and why was it used?",
    "Why did the authors move away from recurrent neural networks?"
]

print(f"Running {len(evaluation_questions)} questions...\n")
print("=" * 60)

results = []
for i, question in enumerate(evaluation_questions):
    result = rag_query(question, collection, embed_model, groq_client)
    results.append(result)
    print(f"\n[{i+1}/20] {question}")
    print(f"Answer: {result['answer'][:200]}...")

print("\n" + "=" * 60)
print("All 20 questions done.")

Running 20 questions...


[1/20] What is the title of the paper?
Answer: I don't have enough information to determine the title of the paper. The provided context appears to be a reprint of various research papers and includes figures and references, but it does not explic...

[2/20] Who are the authors of this paper?
Answer: The authors of the paper "Attention Is All You Need" are:

1. Ashish Vaswani
2. Noam Shazeer
3. Niki Parmar
4. Jakob Uszkoreit
5. Llion Jones
6. Aidan N. Gomez
7. Łukasz Kaiser
8. Illia Polosukhin...

[3/20] What does BLEU score measure?
Answer: The BLEU score measures the quality of a machine translation model by comparing its output to a set of reference translations. It is a metric used to evaluate the fluency and accuracy of the generated...

[4/20] How many attention heads are used in the base model?
Answer: According to the provided context, the base model employs 8 parallel attention layers, or heads....

[5/20] What is the model dimension d_model used in t

In [22]:
def evaluate_answer(question, answer, context_chunks, groq_client):
    context = "\n".join(context_chunks[:3])

    eval_prompt = f"""You are an objective evaluator. Score the answer on two criteria.

Question: {question}
Context used: {context}
Answer given: {answer}

Score on:
1. Relevance (1-3): Does the answer address the question?
   1 = Wrong or off-topic
   2 = Partially answers
   3 = Directly and correctly answers

2. Faithfulness (1-3): Is the answer grounded in the context?
   1 = Hallucinated, not in context
   2 = Partially grounded
   3 = Fully supported by context

Respond ONLY in this exact format, nothing else:
Relevance: X
Faithfulness: X
Reason: one sentence"""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": eval_prompt}],
        temperature=0,
        max_tokens=100
    )
    return response.choices[0].message.content.strip()

print("Evaluating all 20 answers...\n")
print("=" * 60)

evaluation_report = []
for i, result in enumerate(results):
    eval_output = evaluate_answer(
        result["question"],
        result["answer"],
        result["source_chunks"],
        groq_client
    )
    evaluation_report.append({
        "question": result["question"],
        "answer": result["answer"],
        "evaluation": eval_output
    })
    print(f"Q{i+1}: {result['question'][:55]}...")
    print(f"     {eval_output}\n")

Evaluating all 20 answers...

Q1: What is the title of the paper?...
     Relevance: 3
Faithfulness: 3
Reason: The answer directly and correctly addresses the question by stating that the provided context does not explicitly mention the title of the paper, and it is also fully supported by the context as it accurately describes the content of the given text.

Q2: Who are the authors of this paper?...
     Relevance: 3
Faithfulness: 3
Reason: The answer directly and correctly addresses the question by listing the authors of the paper "Attention Is All You Need" as mentioned in the provided context.

Q3: What does BLEU score measure?...
     Relevance: 3
Faithfulness: 3
Reason: The answer directly and correctly addresses the question by explaining what the BLEU score measures, and it is fully supported by the context provided in the paper.

Q4: How many attention heads are used in the base model?...
     Relevance: 3
Faithfulness: 3
Reason: The answer directly quotes the relevant sentenc

In [23]:
relevance_scores = []
faithfulness_scores = []

for item in evaluation_report:
    lines = item["evaluation"].split("\n")
    try:
        rel_line = [l for l in lines if l.strip().startswith("Relevance:")][0]
        faith_line = [l for l in lines if l.strip().startswith("Faithfulness:")][0]

        rel = int(rel_line.split(":")[1].strip()[0])
        faith = int(faith_line.split(":")[1].strip()[0])

        relevance_scores.append(rel)
        faithfulness_scores.append(faith)
    except:
        print(f"Could not parse: {item['question'][:40]}")

avg_relevance = sum(relevance_scores) / len(relevance_scores)
avg_faithfulness = sum(faithfulness_scores) / len(faithfulness_scores)

print("=" * 50)
print("       RAG EVALUATION SUMMARY")
print("=" * 50)
print(f"  Total questions tested : 20")
print(f"  Avg Relevance          : {avg_relevance:.2f} / 3.0")
print(f"  Avg Faithfulness       : {avg_faithfulness:.2f} / 3.0")
print(f"\n  Score distribution:")
for score in [3, 2, 1]:
    rel_count = relevance_scores.count(score)
    faith_count = faithfulness_scores.count(score)
    print(f"    Score {score} → Relevance: {rel_count}/20 | Faithfulness: {faith_count}/20")

print(f"\n  Overall RAG Quality: ", end="")
overall = (avg_relevance + avg_faithfulness) / 2
if overall >= 2.8:
    print("STRONG ✓")
elif overall >= 2.4:
    print("DECENT — room to improve")
else:
    print("WEAK — chunking or prompt needs fixing")

print("=" * 50)

       RAG EVALUATION SUMMARY
  Total questions tested : 20
  Avg Relevance          : 3.00 / 3.0
  Avg Faithfulness       : 3.00 / 3.0

  Score distribution:
    Score 3 → Relevance: 20/20 | Faithfulness: 20/20
    Score 2 → Relevance: 0/20 | Faithfulness: 0/20
    Score 1 → Relevance: 0/20 | Faithfulness: 0/20

  Overall RAG Quality: STRONG ✓



# Basic RAG — Q&A Chatbot on "Attention Is All You Need"

## What This Is
A RAG (Retrieval Augmented Generation) pipeline built from scratch — no LangChain.
Answers questions about the "Attention Is All You Need" paper using semantic search + Groq LLM.

## Stack
| Component | Tool |
|---|---|
| PDF Parsing | pypdf |
| Chunking | Custom sentence-aware (regex) |
| Embeddings | all-MiniLM-L6-v2 (HuggingFace) |
| Vector Store | ChromaDB (persistent) |
| LLM | Groq — llama-3.1-8b-instant |

## Pipeline
PDF → Extract Text → Sentence Chunks → Embed → ChromaDB
Question → Embed → Similarity Search → Top-5 Chunks → Groq → Answer

## Evaluation Results
- 20 questions tested across factual, conceptual, and inference categories
- Avg Relevance: 3.0 / 3.0
- Avg Faithfulness: 3.0 / 3.0
- Evaluation method: LLM-as-judge (llama-3.1-8b-instant)

## Key Design Decisions
- **chunk_size=800, overlap=200** — academic papers need larger chunks to preserve complete ideas
- **Sentence-aware chunking** — prevents sentences being cut mid-way, critical for retrieval quality
- **Garbage chunk filter** (words > 15) — removes PDF extraction artifacts before they reach the LLM
- **Relaxed prompt** — "extract and summarize" instead of "only use context" stops LLM from refusing indirect answers

## Known Flaws
1. **LLM judges itself** — judge and answer model are the same (llama-3.1-8b-instant), scores are optimistically biased
2. **Q1 (paper title) failed** — title lives in PDF metadata, not body text; pypdf misses it
3. **Character-level overlap** — overlap is still character-based not sentence-based, context boundary not perfect
4. **No reranking** — top-5 chunks sent as-is; a reranker would filter irrelevant chunks before LLM sees them
5. **Single PDF only** — no multi-document support, no metadata filtering by source
6. **Retrieval scores were weak (0.55+)** on some questions — MiniLM struggles with technical academic phrasing

## What Would Make This Production-Ready
- Stronger judge model (GPT-4 class) for honest evaluation
- Hybrid search (BM25 + semantic) for better retrieval on technical terms
- Reranking layer before LLM generation
- Chunk metadata (page number, section header) for source attribution
